# Chapter 22 – Worked examples

This notebook runs the examples of [Chapter 22](../index.md) live. The code lives in the modules next to it
(`action_count.py`, `bridge_lookup.py`, `loadflow_limits.py`); the cells only call them, so a change in a module
is a change here too.

Run it from this folder (the kernel's working directory must be `src/`), or open it in Jupyter Lab from the
repository. The ToOp pipeline itself is in `lab_toop.ipynb`, which needs ToOp's interpreter.

In [ ]:
import action_count
import bridge_lookup
import loadflow_limits
import numpy as np

## How big is the action space of one station?

`assignments(d)` enumerates the integers $0 \dots 2^{d-1}-1$ as bit patterns, flips the patterns with more than
$d/2$ branches on busbar B and drops those with exactly one branch on either busbar. The count matches
$2^{d-1} - d$, the unsplit configuration included.

In [ ]:
for d in range(4, 9):
    print(f'd = {d}: {action_count.n_actions(d):4d} actions, closed form {2 ** (d - 1) - d:4d}')
print('the four assignments of a 4-branch station (True = busbar B):')
print(action_count.assignments(4).astype(int))

Random clipping starts when $2^{d-1} > 2^{23}$, at $d = 25$. Drawing $k = 2^{23}$ patterns with replacement
from $N = 2^{24}$ leaves about $N(1 - e^{-k/N})$ distinct ones, 6.6 million rather than 8.4 million.

In [ ]:
n, k = 2**24, 2**23
print(f'expected distinct patterns: {n * (1 - np.exp(-k / n)) / 1e6:.2f} million of {k / 1e6:.2f} million draws')

## The bridge lookup

Six branches, one of them a bridge (Exercise 1): the lookup removes every split that leaves the bridge alone with
a single other branch on a busbar.

In [ ]:
enumerated, after_isolation, after_bridge, removed = bridge_lookup.funnel()
print(enumerated, '->', after_isolation, '->', after_bridge, 'removed:', removed)

## Loadflow-based limits

$I_\text{new} = \operatorname{clip}(f I_\text{lf},\ \min(I_\text{old}, I_\text{lf} + m I_\text{old}),\ I_\text{old})$
with $f = 1.2$, $m = 0.05$ for a 1 000 A line (Exercise 3).

In [ ]:
for i_lf in (0.0, 100.0, 250.0, 500.0, 833.3, 1200.0):
    print(f'I_lf = {i_lf:6.1f} A -> I_new = {loadflow_limits.new_limit(i_lf):6.1f} A')